# 🚗 Final Project — Matakuliah Sains Data
## Prediksi Harga Mobil menggunakan CRISP-DM

**Metode:** CRISP-DM (Cross Industry Standard Process for Data Mining)

**Dataset:** Car_sales.xls

---
### Pendahuluan
Sebagai Data Scientist pada perusahaan manufaktur otomotif, project ini bertujuan untuk:
1. Memberikan **rekomendasi spesifikasi** mobil yang laku di pasar
2. Membangun **model prediksi harga** mobil berdasarkan spesifikasinya
3. Menyajikan hasil prediksi harga melalui antarmuka yang mudah digunakan

---
## PHASE 1 — Business Understanding

Perusahaan manufaktur otomotif membutuhkan sistem yang mampu:
- Mengidentifikasi tipe/spesifikasi mobil yang memiliki penjualan tertinggi di pasar
- Memprediksi harga mobil secara akurat berdasarkan atribut/fitur teknis mobil

**Tujuan bisnis:** Membantu tim produksi dan marketing dalam menentukan spesifikasi mobil yang optimal dan penetapan harga yang kompetitif.

**Target:** Membuat model Machine Learning (Linear Regression) yang dapat memprediksi `Price_in_thousands` berdasarkan variabel-variabel spesifikasi teknis mobil.

---
## PHASE 2 — Data Understanding

### Langkah 1 — Memanggil Library yang Diperlukan

In [ ]:
# Import seluruh library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Konfigurasi tampilan
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('✅ Semua library berhasil dimuat!')

### Langkah 2 — Load Data

In [ ]:
# Muat dataset
# Catatan: File Car_sales.xls sebenarnya berformat CSV, sehingga dibaca dengan pd.read_csv
df = pd.read_csv('Car_sales.xls')

print(f'✅ Data berhasil dimuat!')
print(f'   Jumlah baris : {df.shape[0]}')
print(f'   Jumlah kolom : {df.shape[1]}')

### Langkah 3 — Melihat Data

In [ ]:
# Tampilkan 5 baris pertama
print('=== 5 Baris Pertama Dataset ===')
df.head()

In [ ]:
# Info tipe data dan jumlah non-null
print('=== Informasi Dataset ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
df.describe().round(2)

In [ ]:
# Daftar kolom dan keterangannya
kolom_info = {
    'Manufacturer'        : 'Nama produsen / merek mobil',
    'Model'               : 'Nama model mobil',
    'Sales_in_thousands'  : 'Jumlah penjualan (ribuan unit)',
    '__year_resale_value' : 'Nilai jual kembali 4 tahun (ribu USD)',
    'Vehicle_type'        : 'Tipe kendaraan (Passenger/Car)',
    'Price_in_thousands'  : 'Harga mobil (ribu USD) — TARGET',
    'Engine_size'         : 'Ukuran mesin (liter)',
    'Horsepower'          : 'Tenaga kuda (HP)',
    'Wheelbase'           : 'Jarak sumbu roda (inci)',
    'Width'               : 'Lebar kendaraan (inci)',
    'Length'              : 'Panjang kendaraan (inci)',
    'Curb_weight'         : 'Berat kendaraan (ton)',
    'Fuel_capacity'       : 'Kapasitas tangki bahan bakar (galon)',
    'Fuel_efficiency'     : 'Efisiensi bahan bakar (mpg)',
    'Latest_Launch'       : 'Tanggal peluncuran terbaru',
    'Power_perf_factor'   : 'Faktor performa daya'
}

print('=== Keterangan Kolom ===')
for col, desc in kolom_info.items():
    print(f'  {col:<25} : {desc}')

---
## PHASE 3 — Data Preparation

### Langkah 4 — Menghapus Missing Value

Kita identifikasi kolom mana yang memiliki missing value, kemudian tentukan strategi penanganannya.

In [ ]:
# Hitung missing value per kolom
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing' : missing,
    'Persentase (%)' : missing_pct
}).query('`Jumlah Missing` > 0').sort_values('Jumlah Missing', ascending=False)

print('=== Kolom dengan Missing Value ===')
print(missing_df)
print(f'\nTotal baris data    : {len(df)}')
print(f'Total kolom missing : {len(missing_df)}')

In [ ]:
# Visualisasi missing value
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v > 10 else '#f39c12' if v > 1 else '#3498db'
          for v in missing_df['Persentase (%)']]

bars = ax.barh(missing_df.index, missing_df['Persentase (%)'], color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, missing_df['Jumlah Missing']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val} baris', va='center', fontsize=10)

ax.set_xlabel('Persentase Missing Value (%)', fontsize=12)
ax.set_title('Distribusi Missing Value per Kolom', fontsize=14, fontweight='bold')
ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='Threshold 5%')
ax.legend()
plt.tight_layout()
plt.show()

print('\n📌 Analisis:')
print('  - __year_resale_value memiliki missing value terbanyak (22.9%) → akan diisi dengan median')
print('  - Kolom lain memiliki missing value < 2% → akan diisi dengan median (numerik)')
print('  - Tidak ada kolom yang perlu DIHAPUS karena missing value < 50%')

In [ ]:
# Hapus baris duplikat jika ada
jumlah_duplikat = df.duplicated().sum()
print(f'Jumlah baris duplikat : {jumlah_duplikat}')

if jumlah_duplikat > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'✅ {jumlah_duplikat} baris duplikat berhasil dihapus')
else:
    print('✅ Tidak ada duplikat — data bersih')

### Langkah 5 — Mengisi Missing Value

Strategi pengisian:
- **Kolom numerik** → diisi dengan nilai **median** (lebih robust terhadap outlier dibanding mean)
- **Kolom kategorikal** → diisi dengan nilai **modus** (nilai yang paling sering muncul)

In [ ]:
# Pisahkan kolom numerik dan kategorikal
kolom_numerik = df.select_dtypes(include=[np.number]).columns.tolist()
kolom_kategori = df.select_dtypes(include=['object']).columns.tolist()

print(f'Kolom numerik    : {kolom_numerik}')
print(f'Kolom kategorikal: {kolom_kategori}')

In [ ]:
# Catat nilai median sebelum mengisi
print('=== Nilai Median yang Digunakan untuk Mengisi Missing Value ===')
for col in kolom_numerik:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        print(f'  {col:<30}: median = {med:.3f}')

# Isi missing value kolom numerik dengan median
for col in kolom_numerik:
    df[col] = df[col].fillna(df[col].median())

# Isi missing value kolom kategorikal dengan modus
for col in kolom_kategori:
    if df[col].isnull().sum() > 0:
        modus = df[col].mode()[0]
        df[col] = df[col].fillna(modus)
        print(f'  {col:<30}: modus = {modus}')

# Verifikasi
print('\n=== Verifikasi: Missing Value Setelah Pengisian ===')
sisa_missing = df.isnull().sum().sum()
print(f'Total missing value tersisa: {sisa_missing}')
if sisa_missing == 0:
    print('✅ Semua missing value berhasil diisi!')
else:
    print(df.isnull().sum()[df.isnull().sum() > 0])

### Langkah 6 — Eksplorasi Data (EDA)

#### 6a. 10 Jenis Mobil dengan Jumlah Penjualan Terbanyak

In [ ]:
# Buat kolom nama lengkap
df['Car_Name'] = df['Manufacturer'] + ' ' + df['Model']

# Ambil top 10 berdasarkan penjualan
top10 = df.nlargest(10, 'Sales_in_thousands').reset_index(drop=True)

# Warna gradien berdasarkan ranking
colors = plt.cm.YlOrRd_r(np.linspace(0.2, 0.9, 10))

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(top10['Car_Name'][::-1], top10['Sales_in_thousands'][::-1],
               color=colors, edgecolor='white', linewidth=0.8, height=0.65)

for bar, val in zip(bars, top10['Sales_in_thousands'][::-1]):
    ax.text(bar.get_width() + 4, bar.get_y() + bar.get_height()/2,
            f'{val:,.1f}K unit', va='center', fontsize=10.5, fontweight='bold')

ax.set_xlabel('Jumlah Penjualan (ribuan unit)', fontsize=12)
ax.set_title('🏆 10 Mobil dengan Penjualan Terbanyak', fontsize=15, fontweight='bold', pad=15)
ax.set_xlim(0, 640)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}K'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\n📌 Analisis Penjualan:')
print('  ▸ Ford F-Series mendominasi pasar dengan penjualan 540.6K unit — hampir 2x lipat peringkat ke-2')
print('  ▸ Ford Explorer (276.7K) dan Toyota Camry (248K) menduduki posisi 2 dan 3')
print('  ▸ Merek Ford sangat mendominasi: F-Series, Explorer, Taurus, Ranger, dan Focus masuk Top 10')
print('  ▸ Honda hadir dengan Accord dan Civic, Dodge dengan Ram Pickup dan Caravan')
print('  ▸ Tren pasar menunjukkan preferensi terhadap mobil dengan harga terjangkau (pickup & sedan)')

#### 6b. Harga dari 10 Jenis Mobil dengan Penjualan Terbanyak

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Chart 1: Bar chart harga ---
ax1 = axes[0]
colors_price = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
bars = ax1.bar(range(10), top10['Price_in_thousands'],
               color=colors_price, edgecolor='white', linewidth=0.8, width=0.65)

for i, (bar, val) in enumerate(zip(bars, top10['Price_in_thousands'])):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'${val:.1f}K', ha='center', fontsize=9.5, fontweight='bold')

ax1.set_xticks(range(10))
ax1.set_xticklabels(top10['Car_Name'], rotation=40, ha='right', fontsize=9)
ax1.set_ylabel('Harga (ribu USD)', fontsize=11)
ax1.set_title('💰 Harga 10 Mobil Terlaris', fontsize=13, fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x)}K'))
ax1.spines[['top','right']].set_visible(False)

# --- Chart 2: Scatter Sales vs Price ---
ax2 = axes[1]
scatter = ax2.scatter(top10['Sales_in_thousands'], top10['Price_in_thousands'],
                      s=200, c=top10['Horsepower'], cmap='coolwarm',
                      edgecolors='black', linewidth=0.8, zorder=5)

for _, row in top10.iterrows():
    ax2.annotate(row['Car_Name'], (row['Sales_in_thousands'], row['Price_in_thousands']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)

plt.colorbar(scatter, ax=ax2, label='Horsepower (HP)')
ax2.set_xlabel('Penjualan (ribuan unit)', fontsize=11)
ax2.set_ylabel('Harga (ribu USD)', fontsize=11)
ax2.set_title('📊 Penjualan vs Harga (warna = HP)', fontsize=13, fontweight='bold')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

print('\n📌 Analisis Harga:')
print('  ▸ Ford Explorer memiliki harga tertinggi ($31.9K) di antara 10 besar')
print('  ▸ Ford Ranger memiliki harga terendah ($12.1K) — membuktikan bahwa harga terjangkau = penjualan tinggi')
print('  ▸ Tidak ada korelasi linier kuat antara harga dan jumlah penjualan')
print('  ▸ Mobil dengan harga $12K–$20K mendominasi penjualan tertinggi')
print('  ▸ Scatter plot menunjukkan F-Series (HP sedang, harga menengah) menjadi outlier penjualan')

#### 6c. Atribut Lain: Horsepower, Fuel Efficiency, Engine Size dari 10 Mobil Terlaris

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
car_labels = [f"{r['Manufacturer']}\n{r['Model']}" for _, r in top10.iterrows()]
x = np.arange(10)
width = 0.65

# --- Chart 1: Horsepower ---
ax = axes[0]
c1 = plt.cm.Oranges(np.linspace(0.4, 0.95, 10))
bars1 = ax.bar(x, top10['Horsepower'], color=c1, width=width, edgecolor='white')
for bar, v in zip(bars1, top10['Horsepower']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5, f'{int(v)}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(car_labels, fontsize=7.5, rotation=30, ha='right')
ax.set_ylabel('Horsepower (HP)', fontsize=11)
ax.set_title('⚡ Tenaga Kuda (HP)', fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# --- Chart 2: Fuel Efficiency ---
ax = axes[1]
c2 = plt.cm.Greens(np.linspace(0.4, 0.95, 10))
bars2 = ax.bar(x, top10['Fuel_efficiency'], color=c2, width=width, edgecolor='white')
for bar, v in zip(bars2, top10['Fuel_efficiency']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{int(v)} mpg', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(car_labels, fontsize=7.5, rotation=30, ha='right')
ax.set_ylabel('Fuel Efficiency (mpg)', fontsize=11)
ax.set_title('⛽ Efisiensi Bahan Bakar', fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# --- Chart 3: Engine Size ---
ax = axes[2]
c3 = plt.cm.Purples(np.linspace(0.4, 0.95, 10))
bars3 = ax.bar(x, top10['Engine_size'], color=c3, width=width, edgecolor='white')
for bar, v in zip(bars3, top10['Engine_size']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03, f'{v:.1f}L', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(car_labels, fontsize=7.5, rotation=30, ha='right')
ax.set_ylabel('Ukuran Mesin (Liter)', fontsize=11)
ax.set_title('🔧 Ukuran Mesin (Engine Size)', fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Atribut Teknis 10 Mobil Terlaris', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📌 Analisis Atribut Teknis:')
print('  ▸ HORSEPOWER: Dodge Ram Pickup tertinggi (230 HP), Honda Civic terendah (106 HP)')
print('    → Truck dan SUV cenderung membutuhkan tenaga lebih besar')
print('  ▸ FUEL EFFICIENCY: Honda Civic paling irit (32 mpg), Dodge Ram Pickup paling boros (17 mpg)')
print('    → Sedan dan hatchback lebih efisien; pickup dan SUV lebih boros')
print('  ▸ ENGINE SIZE: Dodge Ram Pickup terbesar (5.2L), Honda Civic terkecil (1.6L)')
print('    → Mobil dengan mesin besar → HP tinggi → efisiensi rendah (tradeoff performa vs konsumsi)')

In [ ]:
# Heatmap korelasi antar variabel numerik
fig, ax = plt.subplots(figsize=(11, 8))
num_cols = ['Price_in_thousands','Sales_in_thousands','Engine_size','Horsepower',
            'Wheelbase','Width','Length','Curb_weight','Fuel_capacity',
            'Fuel_efficiency','Power_perf_factor']
corr_matrix = df[num_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8.5}, cbar_kws={'shrink': 0.8})

ax.set_title('🔥 Heatmap Korelasi Antar Variabel Numerik', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n📌 Korelasi dengan Price_in_thousands:')
corr_price = corr_matrix['Price_in_thousands'].drop('Price_in_thousands').sort_values(ascending=False)
for col, val in corr_price.items():
    arah = '🔴 positif' if val > 0 else '🔵 negatif'
    kuat = 'KUAT' if abs(val) > 0.5 else 'sedang' if abs(val) > 0.3 else 'lemah'
    print(f'  {col:<25}: {val:+.3f} ({arah}, {kuat})')

### Langkah 7 — Menentukan Variabel untuk Rekomendasi dan Prediksi

Berdasarkan hasil analisis korelasi dan eksplorasi data, variabel yang dipilih sebagai **fitur independen** untuk prediksi harga adalah:

In [ ]:
# Variabel yang dipilih berdasarkan:
# 1. Korelasi tinggi dengan Price_in_thousands
# 2. Relevansi bisnis (spesifikasi teknis yang dapat dikendalikan produsen)
# 3. Tidak menyebabkan multikolinearitas ekstrem

FITUR = [
    'Engine_size',        # Korelasi 0.63 dengan harga — spesifikasi inti mesin
    'Horsepower',         # Korelasi 0.84 dengan harga — indikator performa utama
    'Wheelbase',          # Korelasi 0.11 — menentukan segmen kendaraan
    'Width',              # Korelasi 0.33 — dimensi fisik kendaraan
    'Length',             # Korelasi 0.16 — panjang bodi kendaraan
    'Curb_weight',        # Korelasi 0.53 — berat berpengaruh pada segmen & biaya produksi
    'Fuel_capacity',      # Korelasi 0.42 — kapasitas tangki
    'Fuel_efficiency',    # Korelasi -0.49 — efisiensi BBM (negatif: makin efisien biasanya lebih murah)
    'Power_perf_factor',  # Korelasi 0.90 — faktor performa paling berkorelasi dengan harga
]

TARGET = 'Price_in_thousands'

print('=== Variabel yang Dipilih ===')
print(f'\n🎯 TARGET (Variabel Dependen):')
print(f'   → {TARGET} (Harga mobil dalam ribuan USD)')
print(f'\n📐 FITUR (Variabel Independen) — {len(FITUR)} variabel:')
for i, f in enumerate(FITUR, 1):
    corr_val = corr_matrix['Price_in_thousands'][f]
    print(f'   {i}. {f:<25} (korelasi dengan harga: {corr_val:+.3f})')

print('\n💡 Rekomendasi Spesifikasi Mobil Terlaris:')
rec = top10[FITUR].median()
print(f'   Engine Size    : {rec["Engine_size"]:.1f} L')
print(f'   Horsepower     : {rec["Horsepower"]:.0f} HP')
print(f'   Fuel Efficiency: {rec["Fuel_efficiency"]:.0f} mpg')
print(f'   Curb Weight    : {rec["Curb_weight"]:.2f} ton')

---
## PHASE 4 — Modeling

### Langkah 8 — Membuat Model Prediksi Harga Mobil (Linear Regression)

#### 8a. Memisahkan Variabel Independen dan Dependen

In [ ]:
# Siapkan dataset final
df_model = df[FITUR + [TARGET]].dropna().reset_index(drop=True)

X = df_model[FITUR]     # Variabel independen (fitur)
y = df_model[TARGET]    # Variabel dependen (target)

print(f'Ukuran X (fitur)  : {X.shape}')
print(f'Ukuran y (target) : {y.shape}')
print(f'\nStatistik target (Price_in_thousands):')
print(f'  Min    : ${y.min():.2f}K')
print(f'  Max    : ${y.max():.2f}K')
print(f'  Mean   : ${y.mean():.2f}K')
print(f'  Median : ${y.median():.2f}K')
print(f'  Std    : ${y.std():.2f}K')

#### 8b. Memisahkan Data Training (80%) dan Testing (20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Data Training  : {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Data Testing   : {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Total data     : {len(X)} baris')

#### 8c. Membuat Model Regresi

In [ ]:
# Inisialisasi dan latih model
model = LinearRegression()
model.fit(X_train, y_train)

print('✅ Model Linear Regression berhasil dilatih!')
print(f'\n=== Koefisien Model ===')
print(f'  Intercept (b0) : {model.intercept_:.4f}')
print(f'\n  Koefisien fitur:')
for feat, coef in zip(FITUR, model.coef_):
    arah = '↑' if coef > 0 else '↓'
    print(f'  {feat:<25}: {coef:+.4f}  {arah}')

print('\n📌 Interpretasi Koefisien:')
print('  Setiap peningkatan 1 unit pada fitur akan mengubah harga mobil')
print('  sebesar nilai koefisiennya (dalam ribuan USD), dengan variabel lain tetap.')

#### 8d. Evaluasi Model dengan RMSE dan R² Score

In [ ]:
# Prediksi pada data training dan testing
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

# Hitung metrik evaluasi
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test, y_pred_test)

print('=' * 45)
print('          EVALUASI MODEL REGRESI LINIER')
print('=' * 45)
print(f'  {'Metrik':<20} {'Training':>10} {'Testing':>10}')
print(f'  {'-'*40}')
print(f'  {'RMSE (ribu USD)':<20} {rmse_train:>10.4f} {rmse_test:>10.4f}')
print(f'  {'R² Score':<20} {r2_train:>10.4f} {r2_test:>10.4f}')
print('=' * 45)

print(f'\n📌 Interpretasi:')
print(f'  ▸ R² Testing = {r2_test:.4f} → model menjelaskan {r2_test*100:.1f}% variasi harga')
print(f'  ▸ RMSE Testing = ${rmse_test:.2f}K → rata-rata error prediksi sekitar ${rmse_test*1000:,.0f}')

if r2_test >= 0.85:
    print('  ▸ ✅ Model SANGAT BAIK — R² ≥ 0.85')
elif r2_test >= 0.70:
    print('  ▸ ✅ Model BAIK — R² ≥ 0.70')
elif r2_test >= 0.50:
    print('  ▸ ⚠️ Model CUKUP — R² ≥ 0.50, bisa ditingkatkan')
else:
    print('  ▸ ❌ Model KURANG BAIK — pertimbangkan feature engineering atau model lain')

#### 8e. Scatter Plot Hasil Evaluasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, y_true, y_pred, label, color in [
    (axes[0], y_train, y_pred_train, 'Training', '#2ecc71'),
    (axes[1], y_test,  y_pred_test,  'Testing',  '#3498db')
]:
    # Scatter: Actual vs Predicted
    ax.scatter(y_true, y_pred, alpha=0.65, color=color, edgecolors='white',
               linewidth=0.6, s=80, label='Prediksi')

    # Garis ideal (perfect prediction)
    lims = [min(y_true.min(), y_pred.min())-2, max(y_true.max(), y_pred.max())+2]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Prediksi Sempurna', zorder=5)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Harga Aktual (ribu USD)', fontsize=11)
    ax.set_ylabel('Harga Prediksi (ribu USD)', fontsize=11)
    ax.set_title(f'Scatter Plot — Data {label}\nR²={r2:.4f}  |  RMSE=${rmse:.2f}K',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    # Tambahkan teks info
    ax.text(0.05, 0.92, f'R² = {r2:.4f}', transform=ax.transAxes,
            fontsize=11, color='darkred', fontweight='bold')
    ax.text(0.05, 0.84, f'RMSE = ${rmse:.2f}K', transform=ax.transAxes,
            fontsize=11, color='navy', fontweight='bold')

plt.suptitle('Evaluasi Model: Actual vs Predicted Price', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('📌 Semakin dekat titik-titik ke garis merah putus-putus, semakin akurat prediksi model.')

---
## PHASE 5 — Evaluation

### Langkah 9 — Memprediksi Harga dengan Spesifikasi Tertentu

In [ ]:
# Buat DataFrame spesifikasi mobil yang ingin diprediksi
# Spesifikasi diambil dari median 10 mobil terlaris (rekomendasi pasar)
spesifikasi_mobil = pd.DataFrame({
    'Engine_size'       : [2.5],   # Ukuran mesin 2.5L (menengah, efisien)
    'Horsepower'        : [150],   # 150 HP (performa baik, harga terjangkau)
    'Wheelbase'         : [107],   # Jarak sumbu roda 107 inci
    'Width'             : [70],    # Lebar 70 inci
    'Length'            : [185],   # Panjang 185 inci
    'Curb_weight'       : [3.2],   # Berat 3.2 ton
    'Fuel_capacity'     : [16],    # Kapasitas tangki 16 galon
    'Fuel_efficiency'   : [24],    # 24 mpg
    'Power_perf_factor' : [75],    # Faktor performa 75
})

print('=== Spesifikasi Mobil yang Akan Diprediksi ===')
for col in FITUR:
    print(f'  {col:<25}: {spesifikasi_mobil[col].values[0]}')

### Langkah 10 — Menampilkan Hasil Prediksi Harga

In [ ]:
# Prediksi harga
harga_prediksi = model.predict(spesifikasi_mobil)[0]

print('=' * 50)
print('      HASIL PREDIKSI HARGA MOBIL')
print('=' * 50)
print(f'\n  💰 Perkiraan Harga : ${harga_prediksi:.3f}K')
print(f'  💵 Dalam USD       : ${harga_prediksi * 1000:,.0f}')
print(f'  💵 Dalam IDR (est.): Rp {harga_prediksi * 1000 * 16000:,.0f}')
print('\n' + '=' * 50)

# Visualisasi prediksi
fig, ax = plt.subplots(figsize=(10, 5))

# Distribusi harga aktual
ax.hist(df[TARGET], bins=30, color='#3498db', alpha=0.65, edgecolor='white', label='Distribusi Harga Aktual')
ax.axvline(harga_prediksi, color='#e74c3c', linewidth=2.5, linestyle='--', label=f'Prediksi: ${harga_prediksi:.1f}K')
ax.axvline(df[TARGET].median(), color='#f39c12', linewidth=1.5, linestyle=':', label=f'Median Aktual: ${df[TARGET].median():.1f}K')

ax.set_xlabel('Harga (ribu USD)', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.set_title('Distribusi Harga Mobil vs Hasil Prediksi', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\n📌 Dengan spesifikasi yang diberikan, model memprediksi harga mobil')
print(f'   sebesar ${harga_prediksi:.2f}K (~${harga_prediksi*1000:,.0f}).')
print(f'   Harga ini berada di segmen {"premium" if harga_prediksi > 30 else "menengah" if harga_prediksi > 18 else "ekonomis"}.')

---
## PHASE 6 — Deployment (Opsional)

### Langkah 11 — Antarmuka Prediksi Harga untuk End User

Berikut adalah simulasi antarmuka sederhana berbasis input interaktif di Google Colab:

In [ ]:
# Fungsi prediksi yang dapat dipanggil ulang
def prediksi_harga_mobil(
    engine_size,
    horsepower,
    wheelbase,
    width,
    length,
    curb_weight,
    fuel_capacity,
    fuel_efficiency,
    power_perf_factor
):
    """
    Fungsi untuk memprediksi harga mobil berdasarkan spesifikasi teknis.

    Parameter:
    - engine_size       : Ukuran mesin dalam liter (misal: 2.0)
    - horsepower        : Tenaga kuda dalam HP (misal: 150)
    - wheelbase         : Jarak sumbu roda dalam inci (misal: 105)
    - width             : Lebar dalam inci (misal: 70)
    - length            : Panjang dalam inci (misal: 185)
    - curb_weight       : Berat kendaraan dalam ton (misal: 3.0)
    - fuel_capacity     : Kapasitas tangki dalam galon (misal: 15)
    - fuel_efficiency   : Efisiensi BBM dalam mpg (misal: 25)
    - power_perf_factor : Faktor performa daya (misal: 70)

    Return: Harga prediksi dalam ribuan USD
    """
    input_data = pd.DataFrame([{
        'Engine_size'       : engine_size,
        'Horsepower'        : horsepower,
        'Wheelbase'         : wheelbase,
        'Width'             : width,
        'Length'            : length,
        'Curb_weight'       : curb_weight,
        'Fuel_capacity'     : fuel_capacity,
        'Fuel_efficiency'   : fuel_efficiency,
        'Power_perf_factor' : power_perf_factor
    }])
    harga = model.predict(input_data)[0]
    return harga


# ============================================================
# Contoh penggunaan — ubah nilai di bawah sesuai spesifikasi
# ============================================================
harga = prediksi_harga_mobil(
    engine_size       = 2.0,
    horsepower        = 130,
    wheelbase         = 103,
    width             = 68,
    length            = 180,
    curb_weight       = 2.8,
    fuel_capacity     = 14,
    fuel_efficiency   = 28,
    power_perf_factor = 65
)

print('┌─────────────────────────────────────────┐')
print('│       PREDIKSI HARGA MOBIL              │')
print('├─────────────────────────────────────────┤')
print(f'│  Engine Size     : 2.0 L                │')
print(f'│  Horsepower      : 130 HP               │')
print(f'│  Fuel Efficiency : 28 mpg               │')
print(f'│  Power Perf Fac  : 65                   │')
print('├─────────────────────────────────────────┤')
print(f'│  💰 HARGA ESTIMASI : ${harga:.2f}K          │')
print(f'│  💵 ≈ ${harga*1000:>10,.0f}                    │')
print('└─────────────────────────────────────────┘')

In [ ]:
# Simpan model untuk deployment
import pickle

with open('model_prediksi_harga_mobil.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model berhasil disimpan sebagai: model_prediksi_harga_mobil.pkl')
print('   Model ini dapat dimuat ulang dengan: model = pickle.load(open("model_prediksi_harga_mobil.pkl", "rb"))')

# Ringkasan akhir
print('\n' + '='*55)
print('               RINGKASAN FINAL PROJECT')
print('='*55)
print(f'  Dataset            : Car_sales.xls ({len(df)} baris, 16 kolom)')
print(f'  Algoritma          : Linear Regression (sklearn)')
print(f'  Jumlah Fitur       : {len(FITUR)} variabel')
print(f'  Split Data         : 80% train / 20% test')
print(f'  R² Score (test)    : {r2_test:.4f}')
print(f'  RMSE (test)        : ${rmse_test:.4f}K')
print(f'  Mobil Terlaris     : Ford F-Series (540.6K unit)')
print(f'  Harga Rata-Rata    : ${df[TARGET].mean():.1f}K')
print('='*55)